- https://huggingface.co/mistralai/Ministral-3-14B-Reasoning-2512 <-- The model page with specific guidance
- https://docs.langchain.com/oss/python/integrations/chat/vllm  <-- Told by the warning to go read this
- https://docs.langchain.com/oss/python/integrations/chat#chat-completions-api <-- Warns about reasoning limitations
- https://docs.langchain.com/oss/python/integrations/chat/openai <-- Describes how to actually use it
- https://models.dev/?search=Ministral <-- search engine for models!!

# Demo: Vocabulary Middleware

An agent learns about available vocabulary as it builds a graph using middleware tools.

---

- **What is `VocabularyMiddleware`?** A set of tools, prompts, and supporting infrastructure for LangChain agents that use `DatasetMiddleware`
   - **What is `DatasetMiddleware`?** A similar set of utiliies allowing an agent to self-manage a knowledge graph.
     See [the previous demo](./demo-dataset-middleware.ipynb) for more details.

- **What problem does it solve for a Research Agent?** Agents need references with which they can dynamically manage their contexts. The world of RDF Vocabularies is pretty big, semi-structured, and very inconsistently built. While one might be tempted to throw RAG at it, we have some good definitions for what an agent needs to know about a term.

- **What will this notebook demonstrate?** How to give an agent efficient use of existing RDF Vocabularies and ontologies as reference materials.

## 1. Setup

We've got a few things to do before the demo gets started.

First, we have to filter a langchain-internal warning (unrelated to our data model, it's just annoying).

In [12]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE: The only reason for `streaming=False` is that the Ministral model that I'm using starts to embed tool invocations in its normal chat output, causing us to lose tool calls.
Your mileage may vary depending on the model you select._

In [13]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=1,
    # streaming=False,  # NOTE: True -> problems for mistral's tool call responses
    # model_kwargs={
    #    "parallel_tool_calls": False,
    #    "tool_choice": "required",
    # },
)

In [14]:
from typing import Final

SYSTEM_PROMPT: Final[str] = """
You are a research assistant specialized in ontological modeling and the semantic web.
You are given a question to answer or a task to complete.

You have access to tools that allow you to curate a RDF knowledge base, taking advantage of your specialization.
When appropriate, you SHOULD present your answer, reasoning, or supporting evidence in `text/turtle`.

When transforming unstructured text into RDF, you MUST attempt to use controlled RDF vocabularies.
You MUST ensure that your asserted facts are grounded in the provided content; they MUST NOT be assumptions or inferences, unless you have been requested to do so as part of your task.
You MUST NOT assert facts that you are uncertain about given the provided content.
You SHOULD assert all facts that are claimed by the provided content.

You MUST use `add_triples` to add triples to your knowledge base.
You MUST use `serialize_dataset` to serialize your knowledge base before formulating your response.
Your output MUST include the serialized graph as a `text/turtle` block.
"""

In [15]:
TASK: Final[str] = """

Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.


```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```
"""

In [16]:
import logging
import uuid
from typing import Any

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command
from rdflibr.middleware.dataset_middleware import DatasetMiddleware
from rdflibr.middleware.ministral_middleware import MinistralPromptSuffixMiddleware
from rdflibr.middleware.rdf_vocabulary_middleware import RDFVocabularyMiddleware
from rdflibr.middleware.tracing_notebook import LiveNotebookTrace

logging.basicConfig(level=logging.INFO)
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

invoke_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
dataset_middleware = DatasetMiddleware()


with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent: CompiledStateGraph[Any, None, Any, Any] = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                dataset_middleware,
                RDFVocabularyMiddleware(),
                MinistralPromptSuffixMiddleware(),
            ],
            # Human in the loop to approve or reject dataset resets
            interrupt_on={
                "reset_dataset": {"allowed_decisions": ["approve", "reject"]},
            },
            checkpointer=MemorySaver(),
        )
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": TASK}]},
        config=invoke_config,
        version="v2",
    )
    while True:
        if not result.interrupts:
            break
        else:
            interrupt_value = result.interrupts[0].value
            action_requests = interrupt_value["action_requests"]
            review_configs = interrupt_value["review_configs"]

            config_map = {cfg["action_name"]: cfg for cfg in review_configs}

            # Display the pending actions to the user
            for action in action_requests:
                review_config = config_map[action["name"]]
                print(f"Tool: {action['name']}")
                print(f"Arguments: {action['args']}")
                print(f"Allowed decisions: {review_config['allowed_decisions']}")

            # Get user decisions (one per action_request, in order)
            decisions = [
                {"type": "reject"}  # User approved the deletion
            ]
            # Resume execution with decisions
            result = agent.invoke(
                Command(resume={"decisions": decisions}),
                config=invoke_config,  # Must use the same config!
                version="v2",
            )

# from langchain_core.callbacks.stdout import StdOutCallbackHandler
# agent: CompiledStateGraph[Any, None, Any, Any] = create_deep_agent(
#     model=llm,
#     system_prompt=SYSTEM_PROMPT,
#     middleware=[
#         dataset_middleware,
#         RDFVocabularyMiddleware(),
#     ],
#     checkpointer=MemorySaver(),
# )
# result = agent.invoke({"messages": [{"role": "user", "content": TASK}]}, config={"callbacks": [StdOutCallbackHandler()]})

# Dataset Trace

## Turn 1

### Tool: list_triples

#### Arguments

```json
{}
```

#### Result

```json
triples=()
```

## Turn 2

### Tool: list_vocabularies

#### Arguments

```json
{}
```

#### Result

```json
vocabularies=(VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#'), label='RDFS', term_count=15), VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/ns/prov#'), label='PROV-O', term_count=94))
```

## Turn 3

### Model Decision

- Finish reason: `tool_calls`
- Tool calls: `14`

#### Requested Tool Calls

1. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"Person\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"A biological classification for humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

2. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"Homo Sapiens\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"A species of the genus Homo, classified as modern humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:HomoSapiens"
    }
  ]
}
```

3. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:HomoSapiens",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Person"
    }
  ]
}
```

4. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"A subtribe under Hominidae, often including Homo sapiens.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    }
  ]
}
```

5. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:Hominina",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:HomoSapiens"
    }
  ]
}
```

6. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"A family of great apes, including humans and great apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

7. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"Haplorhini\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"An order of primates, including Hominidae and other great apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Haplorhini"
    }
  ]
}
```

8. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:Haplorhini",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae is a subclass of Haplorhini, representing a taxonomic hierarchy.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

9. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:Hominidae",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina is a subtribe under Hominidae.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    }
  ]
}
```

10. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primates"
    },
    {
      "object": "\"Primates\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primates"
    },
    {
      "object": "\"An order of mammals, including prosimians, monkeys, and apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primates"
    }
  ]
}
```

11. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammals"
    },
    {
      "object": "\"Mammals\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammals"
    },
    {
      "object": "\"A class of warm-blooded vertebrate animals, including humans and other animals with hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammals"
    }
  ]
}
```

12. `add_triples`

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animals"
    },
    {
      "object": "\"Animals\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animals"
    },
    {
      "object": "\"A kingdom of multifellular organisms, often controversial in theological discussions, especially regarding humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animals"
    }
  ]
}
```

13. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:Mammals",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Primates"
    }
  ]
}
```

14. `add_triples`

```json
{
  "triples": [
    {
      "object": "urn:ex:Animals",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Mammals"
    }
  ]
}
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:HomoSapiens",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"A subtribe under Hominidae, often including Homo sapiens.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    }
  ]
}
```

#### Result

```json
updated=1 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Hominina",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:HomoSapiens"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"A family of great apes, including humans and great apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

#### Result

```json
updated=1 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Hominidae",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina is a subtribe under Hominidae.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"Person\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"A biological classification for humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Result

```json
updated=2 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"Haplorhini\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"An order of primates, including Hominidae and other great apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Haplorhini"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primates"
    },
    {
      "object": "\"Primates\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primates"
    },
    {
      "object": "\"An order of mammals, including prosimians, monkeys, and apes.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primates"
    }
  ]
}
```

#### Result

```json
updated=1 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Haplorhini",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae is a subclass of Haplorhini, representing a taxonomic hierarchy.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"Homo Sapiens\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"A species of the genus Homo, classified as modern humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:HomoSapiens"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammals"
    },
    {
      "object": "\"Mammals\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammals"
    },
    {
      "object": "\"A class of warm-blooded vertebrate animals, including humans and other animals with hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammals"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animals"
    },
    {
      "object": "\"Animals\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animals"
    },
    {
      "object": "\"A kingdom of multifellular organisms, often controversial in theological discussions, especially regarding humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animals"
    }
  ]
}
```

#### Result

```json
updated=2 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Mammals",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Primates"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Animals",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Mammals"
    }
  ]
}
```

#### Result

```json
updated=1 message='Triples added to the default graph.'
```

## Turn 4

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:John"
    },
    {
      "object": "\"John\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:John"
    },
    {
      "object": "\"An individual person, instance of the Person class.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:John"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 5

### Tool: serialize_dataset

#### Arguments

```json
{
  "format": "turtle"
}
```

#### Result

```json
format='turtle' content='@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<urn:ex:Animals> a rdfs:Class ;\n    rdfs:label "Animals" ;\n    rdfs:comment "A kingdom of multifellular organisms, often controversial in theological discussions, especially regarding humans." .\n\n<urn:ex:Haplorhini> a rdfs:Class ;\n    rdfs:label "Haplorhini" ;\n    rdfs:comment "An order of primates, including Hominidae and other great apes." .\n\n<urn:ex:Hominidae> a rdfs:Class ;\n    rdfs:label "Hominidae" ;\n    rdfs:comment "A family of great apes, including humans and great apes.",\n        "Hominidae is a subclass of Haplorhini, representing a taxonomic hierarchy." ;\n    rdfs:subClassOf <urn:ex:Haplorhini> .\n\n<urn:ex:Hominina> a rdfs:Class ;\n    rdfs:label "Hominina" ;\n    rdfs:comment "A subtribe under Hominidae, often including Homo sapiens.",\n        "Hominina is a subtribe under Hominidae." ;\n    rdfs:subClassOf <urn:ex:Hominidae> .\n\n<urn:ex:HomoSapiens> a rdfs:Class ;\n    rdfs:label "Homo Sapiens" ;\n    rdfs:comment "A species of the genus Homo, classified as modern humans." ;\n    rdfs:subClassOf <urn:ex:Hominina> .\n\n<urn:ex:Mammals> a rdfs:Class ;\n    rdfs:label "Mammals" ;\n    rdfs:comment "A class of warm-blooded vertebrate animals, including humans and other animals with hair or fur." ;\n    rdfs:subClassOf <urn:ex:Animals> .\n\n<urn:ex:Person> a rdfs:Class ;\n    rdfs:label "Person" ;\n    rdfs:comment "A biological classification for humans." ;\n    rdfs:subClassOf <urn:ex:HomoSapiens> .\n\n<urn:ex:Primates> a rdfs:Class ;\n    rdfs:label "Primates" ;\n    rdfs:comment "An order of mammals, including prosimians, monkeys, and apes." ;\n    rdfs:subClassOf <urn:ex:Mammals> .\n\n<urn:ex:John> a <urn:ex:Person> ;\n    rdfs:label "John" ;\n    rdfs:comment "An individual person, instance of the Person class." .\n\n'
```

## Turn 6

### Final Response

> Here is the final RDF graph in Turtle format representing the subject matter of the provided text:
>
> ```text/turtle
> @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
> @prefix ex: <urn:ex:> .
>
> <ex:Animals> a rdfs:Class ;
>     rdfs:label "Animals" ;
>     rdfs:comment "A kingdom of multifellular organisms, often controversial in theological discussions, especially regarding humans." .
>
> <ex:Haplorhini> a rdfs:Class ;
>     rdfs:label "Haplorhini" ;
>     rdfs:comment "An order of primates, including Hominidae and other great apes." .
>
> <ex:Hominidae> a rdfs:Class ;
>     rdfs:label "Hominidae" ;
>     rdfs:comment "A family of great apes, including humans and great apes.",
>         "Hominidae is a subclass of Haplorhini, representing a taxonomic hierarchy." ;
>     rdfs:subClassOf <ex:Haplorhini> .
>
> <ex:Hominina> a rdfs:Class ;
>     rdfs:label "Hominina" ;
>     rdfs:comment "A subtribe under Hominidae, often including Homo sapiens.",
>         "Hominina is a subtribe under Hominidae." ;
>     rdfs:subClassOf <ex:Hominidae> .
>
> <ex:HomoSapiens> a rdfs:Class ;
>     rdfs:label "Homo Sapiens" ;
>     rdfs:comment "A species of the genus Homo, classified as modern humans." ;
>     rdfs:subClassOf <ex:Hominina> .
>
> <ex:Mammals> a rdfs:Class ;
>     rdfs:label "Mammals" ;
>     rdfs:comment "A class of warm-blooded vertebrate animals, including humans and other animals with hair or fur." ;
>     rdfs:subClassOf <ex:Animals> .
>
> <ex:Person> a rdfs:Class ;
>     rdfs:label "Person" ;
>     rdfs:comment "A biological classification for humans." ;
>     rdfs:subClassOf <ex:HomoSapiens> .
>
> <ex:Primates> a rdfs:Class ;
>     rdfs:label "Primates" ;
>     rdfs:comment "An order of mammals, including prosimians, monkeys, and apes." ;
>     rdfs:subClassOf <ex:Mammals> .
>
> <ex:John> a <ex:Person> ;
>     rdfs:label "John" ;
>     rdfs:comment "An individual person, instance of the Person class." .
> ```
>
> This graph captures the hierarchical taxonomy described in the text, the instance of "John" as a person, and the broader relationships between the biological classifications mentioned.


In [17]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

There are 35 triples in the dataset
--------------------------------------------------------------------------------
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Animals> a rdfs:Class ;
    rdfs:label "Animals" ;
    rdfs:comment "A kingdom of multifellular organisms, often controversial in theological discussions, especially regarding humans." .

<urn:ex:Haplorhini> a rdfs:Class ;
    rdfs:label "Haplorhini" ;
    rdfs:comment "An order of primates, including Hominidae and other great apes." .

<urn:ex:Hominidae> a rdfs:Class ;
    rdfs:label "Hominidae" ;
    rdfs:comment "A family of great apes, including humans and great apes.",
        "Hominidae is a subclass of Haplorhini, representing a taxonomic hierarchy." ;
    rdfs:subClassOf <urn:ex:Haplorhini> .

<urn:ex:Hominina> a rdfs:Class ;
    rdfs:label "Hominina" ;
    rdfs:comment "A subtribe under Hominidae, often including Homo sapiens.",
        "Hominina is a subtribe under Hominidae." ;
    rdfs:subClassO

# Conclusions

Compared to having access to only the basic dataset tools [as in our previous demo](),

   - An agent with web search or URL retrieval capabilities can dig as deep as they need to in order to understand what they are working on. This is especially helpful if an agent seems to struggle with the meaning of semantic graphs.